In [1]:
%pip install matplotlib requests moviepy


Note: you may need to restart the kernel to use updated packages.


# V49: full-history credit for the scalar orientation marker

This three-arm CUDA run compares V48a's last-16-step gradient with a full
64-step gradient, and includes a matched blank-seed full-gradient control.
Initialization, fixed masks, target, objective, optimizer, and 150 updates
are shared. Checkpoints are chosen only on the fixed history. Eight unopened
mask histories and one paired D4 path then test transfer and symmetry.

Upload `v17a_frozen_v14a_d4_readout_bridge_artifacts.tar.gz` to `/workspace`.
Run cells in order and download `v49_full_history_marker_credit_artifacts.tar.gz`.
The archive is refreshed after each 25-update boundary; a rerun resumes
automatically from the last boundary. Keep the archive outside Git.


In [2]:
VERSION = 'V49'
SLUG = 'v49_full_history_marker_credit'


In [3]:
#@title Remote GPU setup, locked V17a readout, and lizard target
from io import BytesIO
from pathlib import Path
import hashlib
import importlib.util
import json
import os
import random
import subprocess
import sys
import tarfile

WORKSPACE_ROOT = Path('/workspace') if Path('/workspace').is_dir() else Path.cwd()
BRIDGE_NAME = 'v17a_frozen_v14a_d4_readout_bridge_artifacts.tar.gz'
bridge_candidates = [
    WORKSPACE_ROOT / BRIDGE_NAME,
    Path.cwd() / BRIDGE_NAME,
    Path.cwd() / 'notebooks-executed/v17a' / BRIDGE_NAME,
]
BRIDGE_PATH = next((path.resolve() for path in bridge_candidates if path.is_file()), None)
if BRIDGE_PATH is None:
    raise FileNotFoundError('Upload ' + BRIDGE_NAME + ' into ' + str(WORKSPACE_ROOT))

def file_hash(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

with tarfile.open(BRIDGE_PATH) as archive:
    members = {Path(member.name).name: member for member in archive.getmembers()
               if member.isfile()}
    required = {'metrics.json', 'd4_orbit_decoder.pth'}
    if not required <= members.keys():
        raise RuntimeError('V17a artifact is missing: ' + str(sorted(required - members.keys())))
    bridge_metrics = json.loads(archive.extractfile(members['metrics.json']).read())
    if bridge_metrics.get('status') != 'complete':
        raise RuntimeError('The supplied V17a artifact is not complete')
    if bridge_metrics.get('decision') != 'd4_orbit_needed_for_sharp_v14a_state':
        raise RuntimeError('V17a did not select the required D4-orbit branch')
    if not bridge_metrics['sharp_pass'].get('d4_orbit', False):
        raise RuntimeError('The supplied V17a D4-orbit checkpoint did not pass its sharp gate')
    BRIDGE_ROOT = WORKSPACE_ROOT / (SLUG + '_bridge')
    BRIDGE_ROOT.mkdir(parents=True, exist_ok=True)
    BRIDGE_DECODER_PATH = BRIDGE_ROOT / 'd4_orbit_decoder.pth'
    BRIDGE_DECODER_PATH.write_bytes(
        archive.extractfile(members['d4_orbit_decoder.pth']).read())

repo_candidates = [Path.cwd(), WORKSPACE_ROOT / 'Cells2Pixels']
repo_path = next((path.resolve() for path in repo_candidates
                  if (path / 'models/isonca.py').is_file()), None)
if repo_path is None:
    repo_path = (WORKSPACE_ROOT / 'Cells2Pixels').resolve()
    subprocess.run([
        'git', 'clone', '--branch', 'isotropic', '--single-branch',
        'https://github.com/IvanLudvig/Cells2Pixels.git', str(repo_path)], check=True)
os.chdir(repo_path)
if str(repo_path) not in sys.path:
    sys.path.insert(0, str(repo_path))

if importlib.util.find_spec('lpips') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', 'lpips'], check=True)

import matplotlib.pyplot as plt
import numpy as np
import requests
import torch
import torch.nn.functional as F
from PIL import Image
from lpips import LPIPS
from torch.utils.checkpoint import checkpoint

from losses.image_loss import ImageLoss
from losses.invariant_image_loss import InvariantImageLoss
from models.isonca import GradNormIsoGrowingNCA
from models.isotropic_lppn import _sine_mlp
from training.common import normalize_model_grads

if not torch.cuda.is_available():
    raise RuntimeError('A CUDA GPU runtime is required')
device = torch.device('cuda')
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

TARGET_IMAGE_PATH = Path('data/morphology_png/lizard.png')
if not TARGET_IMAGE_PATH.is_file():
    response = requests.get(
        'https://raw.githubusercontent.com/googlefonts/noto-emoji/43bac1a1272f31cedf0d74c2089fba6c7f952276/png/512/emoji_u1f98e.png',
        headers={'User-Agent': 'Cells2Pixels V49'}, timeout=60)
    response.raise_for_status()
    TARGET_IMAGE_PATH.parent.mkdir(parents=True, exist_ok=True)
    Image.open(BytesIO(response.content)).convert('RGBA').save(TARGET_IMAGE_PATH)
TARGET_HASH = file_hash(TARGET_IMAGE_PATH)
if TARGET_HASH != bridge_metrics['config']['target_hash']:
    raise RuntimeError('V17a and V17b lizard target hashes differ')

def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

def cpu_rng_state():
    return dict(
        python=random.getstate(), numpy=np.random.get_state(),
        torch=torch.get_rng_state(), cuda=torch.cuda.get_rng_state_all())

def restore_rng_state(state):
    random.setstate(state['python'])
    np.random.set_state(state['numpy'])
    torch.set_rng_state(state['torch'])
    torch.cuda.set_rng_state_all(state['cuda'])

revision = subprocess.run(
    ['git', 'rev-parse', 'HEAD'], cwd=repo_path, capture_output=True,
    text=True, check=True).stdout.strip()
repository_sources = {name: file_hash(repo_path / name) for name in (
    'models/isonca.py', 'models/isotropic_lppn.py', 'models/siren.py',
    'losses/image_loss.py', 'losses/invariant_image_loss.py',
    'training/common.py')}
OUTPUT_ROOT = WORKSPACE_ROOT / SLUG
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
RESUME_PATH = OUTPUT_ROOT / 'latest_resume.pt'
ARCHIVE_PATH = WORKSPACE_ROOT / (SLUG + '_artifacts.tar.gz')
print('Repository:', repo_path, revision)
print('V17a bridge:', BRIDGE_PATH, file_hash(BRIDGE_PATH)[:12])
print('Output:', OUTPUT_ROOT)
print('GPU:', torch.cuda.get_device_name(0), '| Torch:', torch.__version__)


Repository: /workspace/Cells2Pixels 90e04a0c48e024a7b6844c28ce8aba3c0ff95c75
V17a bridge: /workspace/v17a_frozen_v14a_d4_readout_bridge_artifacts.tar.gz c16185629f3a
Output: /workspace/v49_full_history_marker_credit
GPU: NVIDIA GeForce RTX 2060 SUPER | Torch: 2.11.0+cu128


In [4]:
#@title Fixed-history marker intervention and image objective
CHANNELS = 32
FC_DIM = 256
GRID_SIZE = 96
RENDER_SCALE = 4
BASE_SEED = 48001
STEPS = 150
AGE = 64
GRAD_STEPS = 16
NCA_LR = 3e-5  # V17b1's numerically stable choice
DECODER_LR = 1e-4
OVERFLOW_WEIGHT = 100.0
STATE_ABS_LIMIT = 32.0
DECODER_D4_LIMIT = 1e-5
MARKER_CHANNEL = 4
MARKER_AMPLITUDE = 0.5
assert GRID_SIZE * RENDER_SCALE == 384 and GRAD_STEPS < AGE

image_loss = ImageLoss(
    target_path=str(TARGET_IMAGE_PATH), image_size=(256, 256),
    padding=(64, 64), l1_weight=1.0, l2_weight=1.0,
    lpips_weight=0.0, premultiply_alpha=True, device=device)
alignment_loss = InvariantImageLoss(
    target_path=str(TARGET_IMAGE_PATH), image_size=(256, 256),
    padding=(64, 64), premultiply_alpha=True, aux_type='noaux',
    mirror=True, sharpen=True, l2_weight=1.0,
    include_nca_alpha=True, device=device)
canonical_target = image_loss.target_image.detach()
assert canonical_target.shape == (1, 4, 384, 384)
target_alpha = canonical_target[:, 3:4].clamp(0, 1)
foreground_roi = F.max_pool2d(target_alpha, 65, stride=1, padding=32)
foreground_weight = 0.1 + 0.9 * foreground_roi

marker_rng = torch.Generator(device='cpu').manual_seed(BASE_SEED)
marker_patch = (2 * torch.rand(3, 3, generator=marker_rng) - 1) * MARKER_AMPLITUDE
assert marker_patch.std() > 0.05
mask_rng = torch.Generator(device='cpu').manual_seed(BASE_SEED + 1)
fixed_masks = (torch.rand(AGE, 1, 1, GRID_SIZE, GRID_SIZE,
                          generator=mask_rng) < 0.5).float().to(device)

config = dict(
    protocol='v49_full_history_marker_credit_v1',
    purpose='tiny fitting and gradient control; no generalization claim',
    model='GradNormIsoGrowingNCA', readout='V17a D4OrbitReadout',
    comparison='same initialization, masks, target, optimizer, and update budget; marker versus blank seed',
    marker=dict(channel=MARKER_CHANNEL, amplitude=MARKER_AMPLITUDE,
                patch=marker_patch.tolist()),
    fixed_pose='native unrotated target; no per-step pose selection',
    fixed_masks_sha256=hashlib.sha256(fixed_masks.cpu().numpy().tobytes()).hexdigest(),
    steps=STEPS, age=AGE, grad_steps=GRAD_STEPS,
    nca_lr=NCA_LR, decoder_lr=DECODER_LR,
    target_hash=TARGET_HASH, bridge_hash=file_hash(BRIDGE_PATH),
    repository_revision=revision, repository_sources=repository_sources,
    torch=torch.__version__)
(OUTPUT_ROOT / 'config.json').write_text(json.dumps(config, indent=2))

def make_seed(model, marker='blank'):
    seed = model.seed(1, GRID_SIZE, GRID_SIZE).float()
    if marker == 'random':
        seed[:, MARKER_CHANNEL, 47:50, 47:50] += marker_patch.to(device)
    elif marker == 'rotated':
        seed[:, MARKER_CHANNEL, 47:50, 47:50] += torch.rot90(
            marker_patch.to(device), 1, (0, 1))
    elif marker != 'blank':
        raise ValueError(marker)
    return seed

def image_objective(model, decoder, state):
    render, state_up, _ = render_state(model, decoder, state)
    weighted_l1 = ((render - canonical_target).abs() * foreground_weight).sum() / (
        foreground_weight.sum() * 4)
    shape_l1 = (2 * state_up[:, 3:4] - target_alpha).abs().mean()
    overflow = (state - state.clamp(-1, 1)).abs().mean()
    total = weighted_l1 + shape_l1 + OVERFLOW_WEIGHT * overflow
    return total, dict(roi_l1=weighted_l1, shape_l1=shape_l1,
                       overflow=overflow), render

def axis_ratio(alpha):
    weight = alpha.clamp(0, 1)[0, 0]
    mass = weight.sum().clamp_min(1e-6)
    yy, xx = torch.meshgrid(
        torch.arange(weight.shape[0], device=weight.device),
        torch.arange(weight.shape[1], device=weight.device), indexing='ij')
    x = xx - (weight * xx).sum() / mass
    y = yy - (weight * yy).sum() / mass
    xx2 = (weight * x * x).sum() / mass
    yy2 = (weight * y * y).sum() / mass
    xy = (weight * x * y).sum() / mass
    eigenvalues = torch.linalg.eigvalsh(torch.stack([
        torch.stack([xx2, xy]), torch.stack([xy, yy2])]))
    return float(torch.sqrt(eigenvalues[-1].clamp_min(0) /
                            eigenvalues[0].clamp_min(1e-6)))


In [5]:
#@title Invariant target poses and exact D4 operations
def rotate_nchw(image, angle_deg):
    angle = torch.as_tensor(angle_deg * torch.pi / 180.0,
                            device=image.device, dtype=image.dtype)
    c, s = torch.cos(angle), torch.sin(angle)
    theta = torch.zeros(image.shape[0], 2, 3, device=image.device, dtype=image.dtype)
    theta[:, 0, 0], theta[:, 0, 1] = c, -s
    theta[:, 1, 0], theta[:, 1, 1] = s, c
    grid = F.affine_grid(theta, image.shape, align_corners=False)
    return F.grid_sample(
        image, grid, mode='bilinear', padding_mode='zeros', align_corners=False)

def transform_specs(phase_angle, mirrored):
    specs = []
    for angle in (phase_angle, -phase_angle):
        if mirrored:
            specs.extend([('rot_flip_x', angle), ('rot_flip_y', angle),
                          ('flip_x_rot', angle), ('flip_y_rot', angle)])
        else:
            specs.append(('rot', angle))
    return specs

def apply_transform(image, spec):
    operation, angle = spec
    if operation == 'rot': return rotate_nchw(image, angle)
    if operation == 'rot_flip_x': return rotate_nchw(image, angle).flip(-1)
    if operation == 'rot_flip_y': return rotate_nchw(image, angle).flip(-2)
    if operation == 'flip_x_rot': return rotate_nchw(image.flip(-1), angle)
    if operation == 'flip_y_rot': return rotate_nchw(image.flip(-2), angle)
    raise ValueError(operation)

def apply_inverse(image, spec):
    operation, angle = spec
    if operation == 'rot': return rotate_nchw(image, -angle)
    if operation == 'rot_flip_x': return rotate_nchw(image.flip(-1), -angle)
    if operation == 'rot_flip_y': return rotate_nchw(image.flip(-2), -angle)
    if operation == 'flip_x_rot': return rotate_nchw(image, -angle).flip(-1)
    if operation == 'flip_y_rot': return rotate_nchw(image, -angle).flip(-2)
    raise ValueError(operation)

@torch.no_grad()
def select_specs(rendered, alpha):
    loss_input = torch.cat([rendered, alpha], 1)
    losses = alignment_loss.calc_losses(loss_input.detach())
    phase_n = alignment_loss.polar_target.shape[-1]
    selected = []
    for index in range(len(rendered)):
        best_index = int(losses[index].argmin())
        mirrored = best_index >= phase_n
        angle = 360.0 * (best_index % phase_n) / phase_n
        candidates = []
        for spec in transform_specs(angle, mirrored):
            aligned_render = apply_transform(rendered[index:index + 1].detach(), spec)
            aligned_alpha = apply_transform(alpha[index:index + 1].detach(), spec)
            fit = ((aligned_render - canonical_target).square().mean()
                   + (2.0 * aligned_alpha - canonical_target[:, 3:4]).square().mean())
            candidates.append((float(fit), spec))
        selected.append(min(candidates, key=lambda item: item[0])[1])
    return selected

def native_targets(specs):
    return torch.cat([apply_inverse(canonical_target, spec) for spec in specs])

def d4_apply(tensor, index):
    transformed = torch.rot90(tensor, index % 4, (-2, -1))
    return transformed.flip(-1) if index >= 4 else transformed


In [6]:
#@title V17a's successful ordered-neighborhood D4-orbit readout
class D4OrbitReadout(torch.nn.Module):
    """Local ordered four-cell readout projected over all eight D4 views."""

    def __init__(self, channels=32, scale_factor=4, chunk_size=16384):
        super().__init__()
        self.channels = channels
        self.scale_factor = scale_factor
        self.chunk_size = chunk_size
        self._geometry_cache = {}
        corners = torch.tensor([
            [-1, -1], [-1, 1], [1, -1], [1, 1]], dtype=torch.int64)
        identity = torch.eye(2, dtype=torch.int64)
        rotation = torch.tensor([[0, -1], [1, 0]], dtype=torch.int64)
        reflection = torch.tensor([[1, 0], [0, -1]], dtype=torch.int64)
        matrices, current = [], identity
        for _ in range(4):
            matrices.extend([current, current @ reflection])
            current = rotation @ current
        matrices = torch.stack(matrices)
        permutations = []
        for matrix in matrices:
            transformed = corners @ matrix.T
            permutations.append(torch.stack([
                torch.nonzero((transformed == corner).all(-1), as_tuple=False)[0, 0]
                for corner in corners]))
        self.register_buffer('d4_matrices', matrices, persistent=False)
        self.register_buffer(
            'd4_permutations', torch.stack(permutations), persistent=False)
        self.orbit_phi = _sine_mlp(
            5 * channels + 2, 64, 0, 64, 10.0, 10.0)
        self.rho = _sine_mlp(
            channels + 64, 64, 1, 4, 10.0, 10.0)

    def _geometry(self, height, width, device, dtype):
        key = (height, width, device.type, device.index, dtype)
        cached = self._geometry_cache.get(key)
        if cached is not None:
            return cached
        scale = self.scale_factor
        py = ((torch.arange(height * scale, device=device, dtype=dtype) + 0.5)
              / scale - 0.5)
        px = ((torch.arange(width * scale, device=device, dtype=dtype) + 0.5)
              / scale - 0.5)
        py, px = torch.meshgrid(py, px, indexing='ij')
        y0, x0 = torch.floor(py), torch.floor(px)
        ty, tx = py - y0, px - x0
        neighbor_y = torch.stack([y0, y0, y0 + 1, y0 + 1], -1)
        neighbor_x = torch.stack([x0, x0 + 1, x0, x0 + 1], -1)
        weights = torch.stack([
            (1 - ty) * (1 - tx), (1 - ty) * tx,
            ty * (1 - tx), ty * tx], -1)
        query_yx = torch.stack([ty - 0.5, tx - 0.5], -1)
        indices = (neighbor_y.long().remainder(height) * width
                   + neighbor_x.long().remainder(width))
        result = indices, weights, query_yx
        self._geometry_cache[key] = result
        return result

    def _orbit_descriptor(self, neighbors, interpolated, query_yx):
        batch, point_n, _, channels = neighbors.shape
        descriptor = None
        for matrix, permutation in zip(self.d4_matrices, self.d4_permutations):
            view_neighbors = neighbors.index_select(-2, permutation)
            view_query = query_yx @ matrix.to(query_yx.dtype).T
            view_query = view_query.unsqueeze(0).expand(batch, -1, -1)
            inputs = torch.cat([
                interpolated,
                view_neighbors.reshape(batch, point_n, 4 * channels),
                view_query], -1)
            encoded = self.orbit_phi(inputs)
            descriptor = encoded if descriptor is None else descriptor + encoded
        return descriptor / 8

    def forward(self, state):
        batch, channels, height, width = state.shape
        if channels != self.channels:
            raise ValueError('Expected ' + str(self.channels) + ' channels')
        indices, weights, query_yx = self._geometry(
            height, width, state.device, state.dtype)
        out_height, out_width, _ = indices.shape
        point_n = out_height * out_width
        flat = state.permute(0, 2, 3, 1).reshape(
            batch, height * width, channels)
        neighbors = flat[:, indices.reshape(-1)].reshape(
            batch, point_n, 4, channels)
        interpolated = (
            neighbors * weights.reshape(1, point_n, 4, 1)).sum(-2)
        query_yx = query_yx.reshape(point_n, 2)
        outputs = []
        for start in range(0, point_n, self.chunk_size):
            stop = min(start + self.chunk_size, point_n)
            arguments = (neighbors[:, start:stop], interpolated[:, start:stop],
                         query_yx[start:stop])
            if self.training:
                descriptor = checkpoint(
                    self._orbit_descriptor, *arguments, use_reentrant=False)
            else:
                descriptor = self._orbit_descriptor(*arguments)
            outputs.append(self.rho(torch.cat([
                interpolated[:, start:stop], descriptor], -1)))
        return torch.cat(outputs, 1).reshape(
            batch, out_height, out_width, 4)


In [7]:
#@title Float32 recurrent system and the unchanged V17b image objective
def make_system(load_bridge=True):
    set_all_seeds(BASE_SEED)
    model = GradNormIsoGrowingNCA(
        channels=CHANNELS, fc_dim=FC_DIM, padding='circular',
        update_prob=0.5, device=device, precision=torch.float32).to(device)
    decoder = D4OrbitReadout(
        channels=CHANNELS, scale_factor=RENDER_SCALE).to(device)
    if load_bridge:
        decoder.load_state_dict(torch.load(
            BRIDGE_DECODER_PATH, map_location=device, weights_only=True))
    return model, decoder

def fresh_pool(model, size):
    seed = model.seed(1, GRID_SIZE, GRID_SIZE).detach().cpu().to(torch.float16)
    return seed.repeat(size, 1, 1, 1)

def make_optimizer(model, decoder, nca_lr, decoder_lr):
    return torch.optim.Adam([
        {'params': list(model.parameters()), 'lr': float(nca_lr), 'name': 'nca'},
        {'params': list(decoder.parameters()), 'lr': float(decoder_lr), 'name': 'decoder'},
    ])

def render_state(model, decoder, state):
    state_float = state.float()
    raw = decoder(state_float).permute(0, 3, 1, 2)
    state_up = F.interpolate(
        state_float, scale_factor=RENDER_SCALE,
        mode='bilinear', align_corners=False)
    living = model.get_living_mask(state_up).to(raw.dtype)
    return raw * living, state_up, living


In [8]:
#@title Predeclared arms, exact boundary resume, and fixed-history training
from torch.utils.checkpoint import checkpoint
ARMS = ('marker_truncated', 'marker_full', 'blank_full')
BOUNDARY_EVERY = 25
AUDIT_SEEDS = tuple(range(49201, 49209))
config.pop('grad_steps')
config.update(
    purpose='test whether full credit through the first 48 growth steps makes a scalar marker useful',
    comparison='same initialization, fixed masks, target, optimizer, and update budget; marker/full-history factorial controls',
    arms=list(ARMS), boundary_every=BOUNDARY_EVERY, audit_seeds=list(AUDIT_SEEDS),
    grad_steps_by_arm=dict(marker_truncated=GRAD_STEPS,
                           marker_full=AGE, blank_full=AGE),
    checkpoint_selection='lowest fixed-history objective at 0,25,...,150',
    registered_gates=dict(fixed_loss=0.15, fixed_gain=0.15,
                          fixed_axis_ratio=1.5, marker_ablation=0.05,
                          fresh_loss=0.165, fresh_gain=0.10,
                          fresh_axis_ratio=1.4, fresh_ablation=0.05,
                          d4_max=0.002, state_abs=STATE_ABS_LIMIT))
config_hash = hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()
(OUTPUT_ROOT / 'config.json').write_text(json.dumps(config, indent=2))

def rollout(model, seed, masks, gradient_steps=0):
    state = seed
    split = len(masks) - gradient_steps
    with torch.no_grad():
        for mask in masks[:split]:
            state, _ = model(state, update_mask=mask)
    if gradient_steps == 0:
        return state
    state = state.detach()
    for start in range(split, len(masks), 8):
        chunk = masks[start:min(start + 8, len(masks))]
        def block(value, chunk=chunk):
            for mask in chunk:
                value, _ = model(value, update_mask=mask)
            return value
        state = checkpoint(block, state, use_reentrant=False)
    return state

def evaluate(model, decoder, marker, masks=fixed_masks, with_render=False):
    model.eval(); decoder.eval()
    with torch.no_grad():
        state = rollout(model, make_seed(model, marker), masks)
        loss, parts, render = image_objective(model, decoder, state)
        if not torch.isfinite(state).all() or not torch.isfinite(render).all():
            raise RuntimeError('Nonfinite evaluation state or render')
        result = dict(loss=float(loss), roi_l1=float(parts['roi_l1']),
                      shape_l1=float(parts['shape_l1']),
                      axis_ratio=axis_ratio(render[:, 3:4]),
                      state_max=float(state.abs().max()))
        if result['state_max'] > STATE_ABS_LIMIT:
            raise RuntimeError('Evaluation state exceeded bound')
        return (result, render.detach().cpu()) if with_render else result

def archive_progress(results, status='training'):
    (OUTPUT_ROOT / 'metrics.json').write_text(json.dumps(
        dict(status=status, config=config, results=results), indent=2))
    with tarfile.open(ARCHIVE_PATH, 'w:gz') as archive:
        for path in sorted(OUTPUT_ROOT.iterdir()):
            if path.is_file():
                archive.add(path, arcname=SLUG + '/' + path.name)

def train_arm(name):
    marker = 'random' if name.startswith('marker') else 'blank'
    gradient_steps = GRAD_STEPS if name == 'marker_truncated' else AGE
    resume_path = OUTPUT_ROOT / (name + '_latest.pt')
    selected_path = OUTPUT_ROOT / (name + '_selected.pt')
    set_all_seeds(BASE_SEED + 2)
    model, decoder = make_system()
    optimizer = make_optimizer(model, decoder, NCA_LR, DECODER_LR)
    if resume_path.is_file():
        saved = torch.load(resume_path, map_location=device, weights_only=False)
        if saved['config_hash'] != config_hash or saved['name'] != name:
            raise RuntimeError('Resume checkpoint does not match V49 protocol')
        model.load_state_dict(saved['model'])
        decoder.load_state_dict(saved['decoder'])
        optimizer.load_state_dict(saved['optimizer'])
        restore_rng_state(saved['rng'])
        start, rows, candidates = saved['step'], saved['rows'], saved['candidates']
        print('Resumed', name, 'from', start)
    else:
        start, rows = 0, []
        candidates = [dict(step=0, **evaluate(model, decoder, marker))]
        torch.save(dict(step=0, model=model.state_dict(),
                        decoder=decoder.state_dict()), selected_path)
    for step in range(start + 1, STEPS + 1):
        model.train(); decoder.train()
        state = rollout(model, make_seed(model, marker), fixed_masks, gradient_steps)
        loss, parts, _ = image_objective(model, decoder, state)
        if not torch.isfinite(loss) or not torch.isfinite(state).all():
            raise RuntimeError(name + ' nonfinite at step ' + str(step))
        state_max = float(state.detach().abs().max())
        if state_max > STATE_ABS_LIMIT:
            raise RuntimeError(name + ' exceeded state bound at step ' + str(step))
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        model_norm = float(torch.linalg.vector_norm(torch.stack([
            torch.linalg.vector_norm(p.grad.detach()) for p in model.parameters()
            if p.grad is not None])))
        readout_norm = float(torch.linalg.vector_norm(torch.stack([
            torch.linalg.vector_norm(p.grad.detach()) for p in decoder.parameters()
            if p.grad is not None])))
        if not np.isfinite(model_norm) or not np.isfinite(readout_norm):
            raise RuntimeError(name + ' nonfinite gradient at step ' + str(step))
        if step == 1 and (model_norm <= 0 or readout_norm <= 0):
            raise RuntimeError(name + ' disconnected gradient')
        normalize_model_grads(model)
        optimizer.step()
        rows.append(dict(step=step, loss=float(loss.detach()),
                         roi_l1=float(parts['roi_l1'].detach()),
                         shape_l1=float(parts['shape_l1'].detach()),
                         state_max=state_max, model_grad=model_norm,
                         readout_grad=readout_norm))
        del state, loss
        if step % BOUNDARY_EVERY == 0:
            candidate = dict(step=step, **evaluate(model, decoder, marker))
            previous_best = min(candidates, key=lambda c: (c['loss'], c['step']))
            candidates.append(candidate)
            if (candidate['loss'], step) < (previous_best['loss'], previous_best['step']):
                torch.save(dict(step=step, model=model.state_dict(),
                                decoder=decoder.state_dict()), selected_path)
            torch.save(dict(config_hash=config_hash, name=name, step=step,
                            model=model.state_dict(), decoder=decoder.state_dict(),
                            optimizer=optimizer.state_dict(), rng=cpu_rng_state(),
                            rows=rows, candidates=candidates), resume_path)
            print(name, step, 'loss', round(candidate['loss'], 6),
                  'axis', round(candidate['axis_ratio'], 3))
            yield dict(rows=rows, candidates=candidates,
                       selected_step=min(candidates, key=lambda c: (c['loss'], c['step']))['step'])

results = {}
for arm in ARMS:
    for partial in train_arm(arm):
        results[arm] = partial
        archive_progress(results)
    if arm not in results:
        saved = torch.load(OUTPUT_ROOT / (arm + '_latest.pt'),
                           map_location='cpu', weights_only=False)
        candidates = saved['candidates']
        results[arm] = dict(rows=saved['rows'], candidates=candidates,
                            selected_step=min(candidates,
                                              key=lambda c: (c['loss'], c['step']))['step'])
    archive_progress(results)


marker_truncated 25 loss 0.236972 axis 1.043
marker_truncated 50 loss 0.209687 axis 1.031
marker_truncated 75 loss 0.192447 axis 1.004
marker_truncated 100 loss 0.190137 axis 1.028
marker_truncated 125 loss 0.189195 axis 1.03
marker_truncated 150 loss 0.188927 axis 1.032
marker_full 25 loss 0.234056 axis 1.034
marker_full 50 loss 0.200412 axis 1.005
marker_full 75 loss 0.18943 axis 1.009
marker_full 100 loss 0.189242 axis 1.019
marker_full 125 loss 0.187918 axis 1.029
marker_full 150 loss 0.188212 axis 1.026
blank_full 25 loss 0.2371 axis 1.036
blank_full 50 loss 0.203103 axis 1.015
blank_full 75 loss 0.189301 axis 1.01
blank_full 100 loss 0.188169 axis 1.021
blank_full 125 loss 0.18768 axis 1.031
blank_full 150 loss 0.187088 axis 1.041


In [9]:
#@title Locked fresh-mask and D4 audit; no checkpoint changes afterward
def fresh_masks(seed):
    generator = torch.Generator(device='cpu').manual_seed(seed)
    return (torch.rand(128, 1, 1, GRID_SIZE, GRID_SIZE,
                       generator=generator) < 0.5).float().to(device)

def load_selected(name):
    model, decoder = make_system()
    saved = torch.load(OUTPUT_ROOT / (name + '_selected.pt'),
                       map_location=device, weights_only=False)
    assert saved['step'] == results[name]['selected_step']
    model.load_state_dict(saved['model'])
    decoder.load_state_dict(saved['decoder'])
    model.eval(); decoder.eval()
    return model, decoder

def paired_d4(model, decoder, masks):
    with torch.no_grad():
        seed = make_seed(model, 'random')
        base = rollout(model, seed, masks)
        paired = rollout(model, d4_apply(seed, 1), d4_apply(masks, 1))
        base_render = render_state(model, decoder, base)[0]
        paired_render = render_state(model, decoder, paired)[0]
        return dict(state=float((paired - d4_apply(base, 1)).abs().max()),
                    image=float((paired_render - d4_apply(base_render, 1)).abs().max()))

audit, panels = {}, {}
for arm in ARMS:
    model, decoder = load_selected(arm)
    marker = 'random' if arm.startswith('marker') else 'blank'
    fixed, panel = evaluate(model, decoder, marker, with_render=True)
    blank = evaluate(model, decoder, 'blank')
    fresh = []
    for seed in AUDIT_SEEDS:
        masks = fresh_masks(seed)
        fresh.append(dict(seed=seed,
                          age64=evaluate(model, decoder, marker, masks[:64]),
                          age128=evaluate(model, decoder, marker, masks),
                          blank_age64=evaluate(model, decoder, 'blank', masks[:64])))
    audit[arm] = dict(fixed=fixed, blank=blank, fresh=fresh,
                      d4=paired_d4(model, decoder, fresh_masks(AUDIT_SEEDS[0])[:64]))
    panels[arm] = panel
    del model, decoder
    torch.cuda.empty_cache()

def mean_fresh(arm, key):
    return float(np.mean([row['age64'][key] for row in audit[arm]['fresh']]))

full = audit['marker_full']
truncated = audit['marker_truncated']
blank = audit['blank_full']
fixed_gain = min(1 - full['fixed']['loss'] / truncated['fixed']['loss'],
                 1 - full['fixed']['loss'] / blank['fixed']['loss'])
fresh_loss = mean_fresh('marker_full', 'loss')
fresh_gain = 1 - fresh_loss / min(mean_fresh('marker_truncated', 'loss'),
                                 mean_fresh('blank_full', 'loss'))
marker_ablation = full['blank']['loss'] / full['fixed']['loss'] - 1
fresh_ablation = float(np.mean([
    row['blank_age64']['loss'] / row['age64']['loss'] - 1
    for row in full['fresh']]))
all_bounded = all(
    row[age]['state_max'] <= STATE_ABS_LIMIT
    for arm in ARMS for row in audit[arm]['fresh']
    for age in ('age64', 'age128'))
gate = bool(
    full['fixed']['loss'] <= 0.15 and fixed_gain >= 0.15
    and full['fixed']['axis_ratio'] >= 1.5
    and marker_ablation >= 0.05
    and fresh_loss <= 0.165 and fresh_gain >= 0.10
    and mean_fresh('marker_full', 'axis_ratio') >= 1.4
    and fresh_ablation >= 0.05 and all_bounded
    and full['d4']['state'] < 0.002 and full['d4']['image'] < 0.002)
if gate:
    decision = 'full_history_marker_mechanism_warrants_distributional_training'
elif full['fixed']['loss'] <= 0.15 and full['fixed']['axis_ratio'] >= 1.5:
    decision = 'fixed_marker_fit_does_not_transfer_or_use_marker'
else:
    decision = 'full_history_marker_mechanism_not_supported'

figure, axes = plt.subplots(1, 4, figsize=(15, 4))
images = [canonical_target.detach().cpu()] + [panels[name] for name in ARMS]
for ax, img, title in zip(axes, images, ['target'] + list(ARMS)):
    ax.imshow(img[0].permute(1, 2, 0).clamp(0, 1))
    ax.set_title(title)
    ax.axis('off')
figure.tight_layout()
figure.savefig(OUTPUT_ROOT / 'selected_fixed_fit_panel.png', dpi=130)
plt.close(figure)

summary = dict(status='complete', config=config, results=results, audit=audit,
               comparison=dict(fixed_gain=fixed_gain, fresh_gain=fresh_gain,
                               marker_ablation=marker_ablation,
                               fresh_marker_ablation=fresh_ablation,
                               target_axis_ratio=axis_ratio(target_alpha)),
               decision=decision)
(OUTPUT_ROOT / 'metrics.json').write_text(json.dumps(summary, indent=2))
with tarfile.open(ARCHIVE_PATH, 'w:gz') as archive:
    for path in sorted(OUTPUT_ROOT.iterdir()):
        if path.is_file():
            archive.add(path, arcname=SLUG + '/' + path.name)
print('Decision:', decision)
print('Comparison:', json.dumps(summary['comparison'], indent=2))
print('Download:', ARCHIVE_PATH)


Decision: full_history_marker_mechanism_not_supported
Comparison: {
  "fixed_gain": -0.004434862297927777,
  "fresh_gain": -0.0102676998744895,
  "marker_ablation": 0.00611816294933587,
  "fresh_marker_ablation": 8.143648408115434e-05,
  "target_axis_ratio": 2.2619054317474365
}
Download: /workspace/v49_full_history_marker_credit_artifacts.tar.gz
